In [1]:
from transformers import AutoImageProcessor
import torch.optim as optim
import webdataset as wds
from modules import *

In [2]:
DATASET_PATH = "/home/austen/GeoDataset/dataset_sharded"
BATCH_SIZE = 8
WORKERS = 1
S2_LEVELS = range(3, 7)
S2_LEVEL_WEIGHTS = [1.0, 1.0, 1.0, 1.0]
LEARNING_RATE = 0.0001
PRETRAINED_MODEL_ID = "facebook/convnext-tiny-224"
EPOCHS = 2
CHECKPOINT_PATH = "checkpoints/checkpoint.pt"

In [3]:
processor = AutoImageProcessor.from_pretrained(PRETRAINED_MODEL_ID, use_fast=True)

dataset = GeoWebDataset(
    DATASET_PATH, 
    processor, 
    levels=S2_LEVELS, 
    shuffle=False,
    num_shards_limit = 1
)

model = HierarchicalConvNeXt(
    pretrained_name = PRETRAINED_MODEL_ID,
    num_classes_per_level = dataset.num_classes_list
)

loader = wds.WebLoader(dataset.dataset, num_workers=WORKERS, batch_size=BATCH_SIZE, pin_memory=True)

trainer = Trainer(model, loader)

optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE)

start_epoch = trainer.load_checkpoint(optimizer, CHECKPOINT_PATH)

for epoch in range(start_epoch, EPOCHS):
    print(f"\nEpoch {epoch}")
    
    trainer.train_epoch(optimizer, weights=S2_LEVEL_WEIGHTS)

    trainer.save_checkpoint(optimizer, epoch, CHECKPOINT_PATH)

Loaded checkpoint from epoch 0

Epoch 1


42it [00:09,  4.66it/s]


KeyboardInterrupt: 